# SAT-15 · Box-fusion (WBF) time of the 3-model ensemble  (CPU, ~2 min)

**Attach:** `sat-ext` + `sat-splits` + `sat-results-07a` (YOLO11s-1024) + `sat-results-07f` (YOLO26s) + `sat-results-07h` (YOLO26m).
**Accelerator:** None. Run all.

Measures the Weighted Box Fusion time per test image for the ensemble YOLO26m + YOLO11s-1024 + YOLO26s (seed 42, standard inference).
The printed `ms per image` is the number for paper Section IV-C (fusion time); ensemble latency = single-model inference time + fusion time.

**Input setup** - right panel -> *Add Input* -> search the name -> *Add*. Links (your own Datasets are private: open them from the account that owns them, or share them with this account first):

* `sat-ext` - code (upload the sat_ext folder of the zip): https://www.kaggle.com/datasets/rifat4018/sat-ext
* `sat-splits` - output of notebook 01: https://www.kaggle.com/datasets/rifat4018/sat-splits
* `sat-results-07a`, `sat-results-07f`, `sat-results-07h` - outputs of earlier notebooks (Output tab of that notebook -> *New Dataset*, same name); search them under *Your Datasets*.
* Check the printed input tree in the next cell before running the rest.

In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rollsroyce1/
      /kaggle/input/datasets/rollsroyce1/sat-ext/
        /kaggle/input/datasets/rollsroyce1/sat-ext/sat_ext/
          CITATION.cff
          LICENSE
          README.md
          analysis.py
          audit.py
          blindaudit.py
          ... +15 more files
      /kaggle/input/datasets/rollsroyce1/sat-results-07a/
        yolo11s.pt
        yolo26n.pt
        /kaggle/input/datasets/rollsroyce1/sat-results-07a/results/
        /kaggle/input/datasets/rollsroyce1/sat-results-07a/results_07A/
        /kaggle/input/datasets/rollsroyce1/sat-results-07a/weights/
          grouped__YOLO11s-1024__s42_best.pt
          grouped__YOLO11s-1024__s43_best.pt
          grouped__YOLO11s-1024__s44_best.pt
          grouped__YOLO11s-1280__s42_best.pt
          grouped__YOLO11s-1280__s43_best.pt
          grouped__YOLO11s-1280__s44_best.pt
      /kaggle/input/datasets/rollsroyce1/sat-results-07f/
        yolo12s.pt
   

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rollsroyce1/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
import glob, time, analysis, dataset as ds
found = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "dets_standard.json*"), recursive=True))
print(len(found), "dets_standard files")
WANT = [("YOLO26m", "42"), ("YOLO11s-1024", "42"), ("YOLO26s", "42")]
def pick(name, seed):
    hits = [f for f in found if f"__{name}__s{seed}".lower() in f.lower().replace(os.sep, "__")]
    hits = [f for f in hits if f"__{name}__s{seed}__".lower() not in f.lower()] or hits   # no tagged variants
    print(name, "seed", seed, "->", hits[0] if hits else "NOT FOUND"); return hits[0] if hits else None
files = [pick(n, s) for n, s in WANT]
assert all(files), "a model was not found: attach sat-results-07a, 07f, 07h (see the list above) and re-run"
dets = [utils.load_any_json(f) for f in files]
gt = utils.load_json(os.path.join(ds.find_split_dir("grouped"), "coco", "test.json"))
print("test images:", len(gt["images"]))
_ = analysis._wbf(dets, gt)          # warm-up
ts = []
for _i in range(5):
    t0 = time.perf_counter(); analysis._wbf(dets, gt); ts.append(time.perf_counter() - t0)
import numpy as np
per_img = 1000 * np.array(ts) / len(gt["images"])
print(f"WBF fusion: {per_img.mean():.2f} +/- {per_img.std():.2f} ms per image (CPU, 3 models)")

40 dets_standard files
YOLO26m seed 42 -> /kaggle/input/datasets/rollsroyce1/sat-results-07h/results/grouped__YOLO26m__s42/dets_standard.json
YOLO11s-1024 seed 42 -> /kaggle/input/datasets/rollsroyce1/sat-results-07a/results/grouped__YOLO11s-1024__s42/dets_standard.json
YOLO26s seed 42 -> /kaggle/input/datasets/rollsroyce1/sat-results-07f/results/grouped__YOLO26s__s42/dets_standard.json
test images: 82
ensemble-boxes not installed -> built-in WBF (same algorithm)
ensemble-boxes not installed -> built-in WBF (same algorithm)
ensemble-boxes not installed -> built-in WBF (same algorithm)
ensemble-boxes not installed -> built-in WBF (same algorithm)
ensemble-boxes not installed -> built-in WBF (same algorithm)
ensemble-boxes not installed -> built-in WBF (same algorithm)
WBF fusion: 67.94 +/- 1.42 ms per image (CPU, 3 models)
